In [1]:
from pathlib import Path
from shapely import wkt
from shapely.geometry import box
import json
from pyproj import Transformer
from shapely.geometry import box

import pandas as pd


DATA_DIR = Path("/Users/ivanr/Developer/ai-vineyard-productivity/data/processed")
AOI_CSV = DATA_DIR / "aoi_processed.csv"
OUTPUT_CSV = DATA_DIR / "aoi_bbox.csv"

In [2]:
vinedos_df = pd.read_csv(DATA_DIR / "vinedos_modelo_limpio.csv")
vinedos_df.head()

,prod_row_id,id_lugar_raw,n_ids_en_fila,es_multi_id,campania,fecha_inicio_cosecha,fecha_fin_cosecha,destino_cultivo,produccion_kg,grados_brix,...,variedad_modo,secano_modo,n_variedades_distintas,n_tratamientos_fito,dosis_fito_total,n_aplicaciones_ferti,dosis_ferti_total,yield_kg_ha,kgdegree,kgdegree_ha
0,0,"L170, L179, L186",3,True,2020,2020-09-11,2020-09-11,Vinificación,1880,NaN,...,NaN,NaN,0,0.0,0.0,0.0,0.000000,1412.047469,22748.0,17085.774373
1,1,L178,1,False,2020,2020-09-16,2020-09-16,Vinificación,390,NaN,...,Ull de llebre,Si,1,0.0,0.0,2.0,0.501439,319.462647,5265.0,4312.745740
2,2,"L198, L1111",2,True,2020,2020-09-16,2020-09-16,Vinificación,470,NaN,...,NaN,NaN,0,0.0,0.0,0.0,0.000000,1332.955190,6110.0,17328.417470
3,3,L196,1,False,2020,2020-09-16,2020-09-16,Vinificación,470,NaN,...,Petit Verdot,Si,1,0.0,0.0,3.0,9252.246406,1597.009854,6110.0,20761.128101
4,4,L1102,1,False,2020,2020-09-16,2020-09-16,Vinificación,530,NaN,...,Petit Verdot,Si,1,0.0,0.0,3.0,9252.245557,1558.365187,6890.0,20258.747427


In [3]:
aoi_processed = pd.read_csv(DATA_DIR / "aoi_processed.csv")

In [10]:
aoi_processed["geometry_area"] = aoi_processed["Geometry"].apply(lambda g: wkt.loads(g).area)
aoi_processed = aoi_processed.sort_values("geometry_area", ascending=False).reset_index(drop=True)
aoi_processed

,id_lugar_raw,Provincia,Municipio,Agregado,Zona,Poligono,Parcela,n_recintos,Geometry,geometry_area
0,"L518321, L518322, L518323, L518328, L518329, L...",43,1,0,0,19,33,23,MULTIPOLYGON (((168506.97015832167 5066392.890...,182802.032561
1,"L518317, L518321, L518322, L518323, L518327, L...",43,1,0,0,18,9011,23,MULTIPOLYGON (((168506.97015832167 5066392.890...,182802.032561
2,"L518314, L518317, L518321, L518322, L518323, L...",43,1,0,0,18,9010,23,MULTIPOLYGON (((168506.97015832167 5066392.890...,182802.032561
3,"L518323, L518333, L518307, L518314, L518317, L...",43,1,0,0,19,34,23,MULTIPOLYGON (((168506.97015832167 5066392.890...,182802.032561
4,"L518307, L518314, L518317, L518321, L518322, L...",43,1,0,0,18,4,23,MULTIPOLYGON (((168506.97015832167 5066392.890...,182802.032561
...,...,...,...,...,...,...,...,...,...,...
3836,"L58843, L59763",8,93,0,0,6,9000,1,POLYGON ((191342.48594647818 5067725.202109708...,84.011587
3837,"L59808, L58909, L58911",8,93,0,0,8,6,1,POLYGON ((190390.06568147603 5067806.228460124...,77.222868
3838,"L58909, L58911, L59808",8,93,0,0,8,6,1,POLYGON ((190390.06568147603 5067806.228460124...,77.222868
3839,L59638,8,93,0,0,1,9000,1,"POLYGON ((192046.7565381355 5067544.418685638,...",75.485958


In [20]:
geometry_parcel = aoi_processed[aoi_processed.id_lugar_raw == "L169"].Geometry.values[0]
print(geometry_parcel)

POLYGON ((198024.3425402035 5068657.957424267, 198019.3037796475 5068660.8674300425, 198009.13295161063 5068666.339128676, 198001.51891992858 5068669.050046174, 197991.99200298014 5068668.627118123, 197986.65466026458 5068668.838519968, 197982.40910187672 5068672.146462045, 197976.78250769415 5068673.191036299, 197974.2817888011 5068671.748404671, 197966.145145033 5068671.922500365, 197959.4081984446 5068672.382610428, 197955.75980437585 5068672.046854434, 197952.73656670362 5068674.160873853, 197952.78788551225 5068675.342237846, 197961.01298424517 5068707.7500931015, 197969.02804909024 5068743.814093363, 197969.7651737966 5068745.05764339, 197969.9984411087 5068745.1695628995, 197970.9782571266 5068745.667107317, 197972.35919961447 5068745.779026833, 197973.30626490177 5068745.8785108505, 197977.92514429594 5068745.182122756, 197979.655987752 5068744.883670731, 197981.4895621323 5068744.597654217, 197991.62763943858 5068742.63284534, 197998.08933059862 5068741.401731122, 198018.03433

In [21]:
MARGIN = 500  # meters

biggest_geom = wkt.loads(geometry_parcel)
minx, miny, maxx, maxy = biggest_geom.bounds
bbox = box(minx - MARGIN, miny - MARGIN, maxx + MARGIN, maxy + MARGIN)
bbox_wkt = bbox.wkt
print(bbox_wkt)

# Transform to WGS84 (EPSG:4326) and print as GeoJSON
transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
lon_min, lat_min = transformer.transform(minx - MARGIN, miny - MARGIN)
lon_max, lat_max = transformer.transform(maxx + MARGIN, maxy + MARGIN)

geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [lon_min, lat_max],
                        [lon_min, lat_min],
                        [lon_max, lat_min],
                        [lon_max, lat_max],
                        [lon_min, lat_max],
                    ]
                ],
            },
        }
    ],
}
print(json.dumps(geojson, indent=2))

POLYGON ((198808.37236512513 5068103.414968264, 198808.37236512513 5069245.8785108505, 197452.73656670362 5069245.8785108505, 197452.73656670362 5068103.414968264, 198808.37236512513 5068103.414968264))
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [
          [
            [
              1.7737481114909537,
              41.38466212980594
            ],
            [
              1.7737481114909537,
              41.376961523946825
            ],
            [
              1.78592599506517,
              41.376961523946825
            ],
            [
              1.78592599506517,
              41.38466212980594
            ],
            [
              1.7737481114909537,
              41.38466212980594
            ]
          ]
        ]
      }
    }
  ]
}
